# 🛩️ Aerial Detection — GPU Inference Server

This notebook runs a **FastAPI inference server** on Kaggle's GPU (Tesla T4/P100) and exposes it
to the internet via **Ngrok**. The Streamlit app on Render acts as a thin UI client that forwards
inference requests to this server.

## Architecture
```
User → Render (Streamlit UI) → Ngrok Tunnel → Kaggle GPU (FastAPI + PyTorch/YOLO)
```

## Prerequisites
1. Enable **GPU** (Tesla T4) in Kaggle Session Options
2. Enable **Internet** access in Session Options
3. Add your **NGROK_AUTHTOKEN** in Add-ons → Secrets
4. Attach dataset: `aghasonemmanuel/aerial-bird-drone-detection`

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pyngrok ultralytics python-multipart

## Cell 2 — Verify GPU & Model Weights

In [ ]:
import torch
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

DATASET = "/kaggle/input/aerial-bird-drone-detection"
weights = [
    "models/classification/custom_cnn/best_model.pth",
    "models/classification/resnet50/best_model.pth",
    "models/classification/mobilenet_v2/best_model.pth",
    "models/classification/efficientnet_b0/best_model.pth",
    "models/detection/best.pt",
]
for w in weights:
    path = os.path.join(DATASET, w)
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path)/1e6:.1f}MB" if exists else "MISSING"
    status = "✅" if exists else "❌"
    print(f"  {status} {w} ({size})")

## Cell 3 — Write Inference Server

In [ ]:
%%writefile /kaggle/working/inference_server.py
"""FastAPI inference server for Aerial Object Classification & Detection.

Runs on Kaggle GPU (Tesla T4/P100) and is exposed via Ngrok tunnel.
Serves both classification and detection endpoints.
"""

import io
import os
import base64
import time

import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import transforms, models
from fastapi import FastAPI, File, UploadFile, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

# ── Paths (Kaggle dataset mount) ───────────────────────────────
DATASET_ROOT = "/kaggle/input/aerial-bird-drone-detection"
CLASSIFICATION_DIR = os.path.join(DATASET_ROOT, "models", "classification")
DETECTION_WEIGHTS = os.path.join(DATASET_ROOT, "models", "detection", "best.pt")

CLASS_NAMES = ["Bird", "Drone"]

PREPROCESS = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── FastAPI app ─────────────────────────────────────────────────
app = FastAPI(title="Aerial Detection API", version="1.0.0")

# Allow Render frontend to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Global model cache ──────────────────────────────────────────
_models = {}
_device = None


def get_device():
    global _device
    if _device is None:
        _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[INFO] Using device: {_device}")
    return _device


class AerialCNN(nn.Module):
    """4-block Custom CNN for binary classification."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(True), nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def _load_classifier(backbone: str):
    if backbone == "custom_cnn":
        model = AerialCNN(num_classes=2)
    elif backbone == "resnet50":
        model = models.resnet50(weights=None)
        in_f = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_f, 2))
    elif backbone == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        in_f = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_f, 2))
    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        in_f = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_f, 2))
    else:
        raise ValueError(f"Unknown backbone: {backbone}")

    weights_path = os.path.join(CLASSIFICATION_DIR, backbone, "best_model.pth")
    state = torch.load(weights_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state)
    return model


def get_classifier(backbone: str):
    if backbone not in _models:
        device = get_device()
        model = _load_classifier(backbone)
        model.to(device).eval()
        _models[backbone] = model
        print(f"[INFO] Loaded classifier: {backbone}")
    return _models[backbone]


def get_detector():
    if "yolov8m" not in _models:
        from ultralytics import YOLO
        model = YOLO(DETECTION_WEIGHTS)
        _models["yolov8m"] = model
        print("[INFO] Loaded YOLOv8m detector")
    return _models["yolov8m"]


# ── Endpoints ───────────────────────────────────────────────────

@app.get("/health")
async def health():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return {
        "status": "ok",
        "device": device,
        "models_loaded": list(_models.keys()),
        "timestamp": time.time(),
    }


@app.post("/classify")
async def classify(
    file: UploadFile = File(...),
    backbone: str = Query("mobilenet_v2"),
):
    t0 = time.time()
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    device = get_device()
    model = get_classifier(backbone)
    tensor = PREPROCESS(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).squeeze().cpu()

    pred_idx = probs.argmax().item()
    return JSONResponse(content={
        "prediction": CLASS_NAMES[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
        "probabilities": {CLASS_NAMES[i]: round(float(probs[i]), 4) for i in range(len(CLASS_NAMES))},
        "backbone": backbone,
        "device": str(device),
        "inference_ms": round((time.time() - t0) * 1000, 1),
    })


@app.post("/detect")
async def detect(
    file: UploadFile = File(...),
    confidence: float = Query(0.25, ge=0.1, le=1.0),
):
    t0 = time.time()
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    model = get_detector()
    results = model.predict(image, conf=confidence, verbose=False)
    result = results[0]

    detections = []
    for box in result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        detections.append({
            "class": result.names[cls_id],
            "confidence": round(conf, 4),
            "bbox": [int(x1), int(y1), int(x2), int(y2)],
        })

    # Encode annotated image as base64 JPEG
    annotated_bgr = result.plot()
    annotated_rgb = annotated_bgr[..., ::-1]
    annotated_pil = Image.fromarray(annotated_rgb)
    buf = io.BytesIO()
    annotated_pil.save(buf, format="JPEG", quality=85)
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    return JSONResponse(content={
        "detections": detections,
        "count": len(detections),
        "annotated_image_b64": img_b64,
        "confidence_threshold": confidence,
        "device": str(get_device()),
        "inference_ms": round((time.time() - t0) * 1000, 1),
    })

## Cell 4 — Configure Ngrok & Launch Server

**IMPORTANT:** Add your `NGROK_AUTHTOKEN` in Kaggle → Add-ons → Secrets before running this cell.

Get your free auth token at: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import subprocess
import time
from pyngrok import ngrok, conf

# ── Load Ngrok auth token from Kaggle Secrets ──────────────────
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
NGROK_TOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(NGROK_TOKEN)
print("✅ Ngrok authenticated")

# ── Start FastAPI server in background ─────────────────────────
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "inference_server:app",
     "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
    cwd="/kaggle/working",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(5)  # Wait for server to boot
print(f"✅ FastAPI server started (PID: {server_proc.pid})")

# ── Open Ngrok tunnel ─────────────────────────────────────────
public_url = ngrok.connect(8000, "http")
print(f"\n{'='*60}")
print(f"🌐 PUBLIC API URL: {public_url}")
print(f"{'='*60}")
print(f"\nHealth check:  {public_url}/health")
print(f"Classify:      POST {public_url}/classify")
print(f"Detect:        POST {public_url}/detect")
print(f"API docs:      {public_url}/docs")
print(f"\n⚠️  Copy the URL above and set it as INFERENCE_API_URL")
print(f"    in your Render environment variables.")
print(f"\n⏰ This session will run for up to 12 hours.")

## Cell 5 — Test the API Locally

In [ ]:
import requests

# Test health endpoint
url = str(public_url).replace("NgrokTunnel: ", "").split('"')[1] if '"' in str(public_url) else str(public_url)
# Extract clean URL
if hasattr(public_url, 'public_url'):
    api_url = public_url.public_url
else:
    api_url = str(public_url)

print(f"Testing API at: {api_url}")

# Health check
r = requests.get(f"{api_url}/health")
print(f"\n✅ Health: {r.json()}")

# Test classification with a sample image from the dataset
import glob
test_images = glob.glob("/kaggle/input/aerial-bird-drone-detection/test/images/*.jpg")[:1]
if test_images:
    with open(test_images[0], "rb") as f:
        r = requests.post(f"{api_url}/classify", files={"file": f}, params={"backbone": "mobilenet_v2"})
    print(f"\n✅ Classification: {r.json()}")

    with open(test_images[0], "rb") as f:
        r = requests.post(f"{api_url}/detect", files={"file": f}, params={"confidence": 0.25})
    resp = r.json()
    print(f"\n✅ Detection: {resp['count']} objects found in {resp['inference_ms']}ms")
    for d in resp['detections']:
        print(f"   {d['class']}: {d['confidence']:.1%} at {d['bbox']}")
else:
    print("⚠️ No test images found in dataset")

## Cell 6 — Keep Session Alive

Run this cell to keep the Kaggle session alive for up to **12 hours**.
The server will continue serving requests via the Ngrok tunnel.

**Do NOT stop this cell** — it keeps the tunnel open.

In [ ]:
import time
from datetime import datetime, timedelta

start = datetime.now()
max_runtime = timedelta(hours=11, minutes=50)  # Safety margin before 12h limit

print(f"🟢 Server running since {start.strftime('%H:%M:%S')}")
print(f"🔗 Tunnel URL: {api_url}")
print(f"⏰ Will auto-shutdown at ~{(start + max_runtime).strftime('%H:%M:%S')}")
print(f"\nPress ⬛ Stop to manually terminate.\n")

try:
    while datetime.now() - start < max_runtime:
        elapsed = datetime.now() - start
        hours, remainder = divmod(int(elapsed.total_seconds()), 3600)
        minutes, seconds = divmod(remainder, 60)
        print(f"\r⏱️  Uptime: {hours:02d}:{minutes:02d}:{seconds:02d} | Models cached: {len(_models) if '_models' in dir() else '?'}", end="", flush=True)
        time.sleep(30)
except KeyboardInterrupt:
    print("\n\n🛑 Manually stopped.")
finally:
    ngrok.disconnect(public_url.public_url if hasattr(public_url, 'public_url') else str(public_url))
    server_proc.terminate()
    print("🔴 Server & tunnel shut down cleanly.")